# Telco Customer Churn analysis

This self-contained notebook combines the project scripts. It compares six XGBoost models: full-feature models trained with undersampling, original data, and SMOTE-NC, plus a reduced-feature version of each. Each reduced model drops features with grouped gain importance at or below 0.02 for that training approach.

Place WA_Fn-UseC_-Telco-Customer-Churn.csv beside this notebook. Required packages: pandas, scikit-learn, xgboost, optuna, and imbalanced-learn. If needed, run this in a notebook cell:

    %pip install pandas scikit-learn xgboost optuna imbalanced-learn

Run the notebook cells from top to bottom. It runs 30 Optuna trials with five-fold CV and then fits six models.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from imblearn.over_sampling import SMOTENC
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 42
FEATURE_IMPORTANCE_CUTOFF = 0.02
MIN_CHURN_PRECISION = 0.55


## Load and split the data

The test set is held back for final evaluation. A separate validation set, with the original class proportions, is used to select decision thresholds. Only model-fitting training data is resampled.


In [ ]:
DATA_PATH = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Put the CSV beside the notebook "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].astype("int64")

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train_original, X_val, y_train_original, y_val = train_test_split(
    X_train_pool, y_train_pool, test_size=0.20,
    random_state=RANDOM_STATE, stratify=y_train_pool,
)

print("Original fitting-training label counts:")
print(y_train_original.value_counts().sort_index().rename(index={0: "No churn", 1: "Churn"}))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))


## Define undersampling and SMOTE-NC

SMOTE-NC is used because this dataset has both numeric and categorical features. Neither sampling method touches validation or test rows.


In [ ]:
def undersample_majority_class(X_part, y_part, majority_class=0, random_state=42):
    counts = y_part.value_counts()
    if len(counts) != 2 or majority_class not in counts.index:
        raise ValueError("Expected a binary target containing the majority class.")
    minority_class = counts.drop(index=majority_class).idxmin()
    minority_indices = y_part[y_part == minority_class].index
    majority_indices = y_part[y_part == majority_class].sample(
        n=len(minority_indices), random_state=random_state
    ).index
    indices = list(minority_indices) + list(majority_indices)
    indices = pd.Series(indices).sample(frac=1, random_state=random_state).tolist()
    return X_part.loc[indices], y_part.loc[indices]


def oversample_churn_with_smote(X_part, y_part, churn_class=1, random_state=42):
    counts = y_part.value_counts()
    if len(counts) != 2 or churn_class not in counts.index:
        raise ValueError("Expected a binary target containing the churn class.")
    if counts[churn_class] >= counts.drop(index=churn_class).iloc[0]:
        raise ValueError("Churn is not the minority class.")
    if counts[churn_class] < 2:
        raise ValueError("SMOTE requires at least two churn examples.")

    X_ready = X_part.copy()
    categorical = list(
        X_ready.select_dtypes(include=["object", "category", "bool"]).columns
    )
    for column in X_ready.select_dtypes(include="number").columns:
        if X_ready[column].nunique(dropna=True) <= 2:
            categorical.append(column)

    for column in categorical:
        values = X_ready[column].astype(object)
        X_ready[column] = values.where(values.notna(), "__missing__")
    for column in X_ready.select_dtypes(include="number").columns:
        X_ready[column] = X_ready[column].fillna(X_ready[column].median())

    sampler = SMOTENC(
        categorical_features=categorical,
        random_state=random_state,
        k_neighbors=min(5, int(counts[churn_class]) - 1),
    )
    X_resampled, y_resampled = sampler.fit_resample(X_ready, y_part)
    X_resampled = pd.DataFrame(X_resampled, columns=X_part.columns)
    y_resampled = pd.Series(y_resampled, name=y_part.name)
    return X_resampled, y_resampled


X_train_under, y_train_under = undersample_majority_class(
    X_train_original, y_train_original, majority_class=0, random_state=RANDOM_STATE
)
X_train_smote, y_train_smote = oversample_churn_with_smote(
    X_train_original, y_train_original, churn_class=1, random_state=RANDOM_STATE
)

print("Undersampled training labels:")
print(y_train_under.value_counts().sort_index().rename(index={0: "No churn", 1: "Churn"}))
print("
SMOTE training labels:")
print(y_train_smote.value_counts().sort_index().rename(index={0: "No churn", 1: "Churn"}))


## Build the model pipeline and tune on undersampled data

Imputation and one-hot encoding are fitted inside each cross-validation fold. Optuna tunes ROC AUC on the undersampled fitting data; the same best hyperparameters are reused for the other approaches for an initial sampling comparison.


In [ ]:
def make_model_pipeline(reference_X, feature_columns):
    selected_X = reference_X.loc[:, feature_columns]
    numeric_features = selected_X.select_dtypes(include="number").columns
    categorical_features = selected_X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns

    transformers = []
    if len(numeric_features):
        transformers.append(
            ("numeric", SimpleImputer(strategy="median"), numeric_features)
        )
    if len(categorical_features):
        categorical_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]
        )
        transformers.append(
            ("categorical", categorical_pipeline, categorical_features)
        )

    preprocessor = ColumnTransformer(transformers=transformers)
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="auc",
                    importance_type="gain",
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                ),
            ),
        ]
    )


all_features = X_train_under.columns.tolist()
base_model = make_model_pipeline(X_train_under, all_features)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def objective(trial):
    params = {
        "classifier__n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "classifier__max_depth": trial.suggest_int("max_depth", 2, 8),
        "classifier__learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),
        "classifier__min_child_weight": trial.suggest_float(
            "min_child_weight", 0.5, 10.0, log=True
        ),
        "classifier__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "classifier__colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "classifier__gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "classifier__reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10.0, log=True
        ),
        "classifier__reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 20.0, log=True
        ),
    }
    candidate = clone(base_model).set_params(**params)
    scores = cross_val_score(
        candidate, X_train_under, y_train_under,
        cv=cv, scoring="roc_auc", n_jobs=1,
    )
    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=30)
best_params = {
    f"classifier__{name}": value for name, value in study.best_params.items()
}
print(f"Best mean CV ROC AUC: {study.best_value:.3f}")
print("Best hyperparameters:", study.best_params)


## Fit full-feature models and get gain importance

Importances for one-hot categories are combined back into their original input feature. The cutoff is applied per approach, not summed across models.


In [ ]:
models_full = {
    "Undersampled": clone(base_model).set_params(**best_params).fit(
        X_train_under, y_train_under
    ),
    "Original training data": clone(base_model).set_params(**best_params).fit(
        X_train_original, y_train_original
    ),
    "SMOTE oversampled": clone(base_model).set_params(**best_params).fit(
        X_train_smote, y_train_smote
    ),
}


def original_feature_importance(fitted_model):
    preprocessor = fitted_model.named_steps["preprocessor"]
    classifier = fitted_model.named_steps["classifier"]
    encoded_names = preprocessor.get_feature_names_out()
    numeric_features = set(preprocessor.transformers_[0][2])
    categorical_features = list(preprocessor.transformers_[1][2])

    rows = []
    for encoded_name, importance in zip(
        encoded_names, classifier.feature_importances_
    ):
        name = encoded_name.split("__", maxsplit=1)[-1]
        if name in numeric_features:
            original_name = name
        else:
            original_name = next(
                (col for col in categorical_features if name.startswith(f"{col}_")),
                name,
            )
        rows.append((original_name, importance))
    return (
        pd.DataFrame(rows, columns=["Feature", "Importance"])
        .groupby("Feature")["Importance"]
        .sum()
    )


importance_table = pd.DataFrame(
    {name: original_feature_importance(model) for name, model in models_full.items()}
).fillna(0.0)

print("Gain importance by original feature:")
display(importance_table.round(4))


## Select features separately and fit reduced-feature models

For each full-feature model, remove its own features with grouped importance at or below 0.02, then refit that approach using its own retained features.


In [ ]:
selected_features_by_model = {}
for model_name in models_full:
    kept = importance_table.index[
        importance_table[model_name] > FEATURE_IMPORTANCE_CUTOFF
    ].tolist()
    removed = importance_table.index[
        importance_table[model_name] <= FEATURE_IMPORTANCE_CUTOFF
    ].tolist()
    if not kept:
        raise ValueError(f"No features exceeded the cutoff for {model_name}.")
    selected_features_by_model[model_name] = kept
    print(f"
{model_name}: kept {len(kept)}, removed {len(removed)}")
    print("Kept:", kept)
    print("Removed:", removed)


def fit_reduced_model(model_name, X_fit, y_fit):
    features = selected_features_by_model[model_name]
    pipeline = make_model_pipeline(X_train_under, features)
    return clone(pipeline).set_params(**best_params).fit(X_fit[features], y_fit)


models_reduced = {
    "Undersampled": fit_reduced_model("Undersampled", X_train_under, y_train_under),
    "Original training data": fit_reduced_model(
        "Original training data", X_train_original, y_train_original
    ),
    "SMOTE oversampled": fit_reduced_model(
        "SMOTE oversampled", X_train_smote, y_train_smote
    ),
}


## Compare validation thresholds

Thresholds are selected separately for each model on the unbalanced validation data: maximize churn recall while meeting the configured minimum precision.


In [ ]:
models_to_compare = {
    **models_full,
    **{f"{name}, selected features": model for name, model in models_reduced.items()},
}
feature_key_for_reduced = {
    f"{name}, selected features": name for name in models_reduced
}
threshold_values = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows = []
selected_thresholds = {}

for model_name, fitted_model in models_to_compare.items():
    source_name = feature_key_for_reduced.get(model_name)
    features = selected_features_by_model[source_name] if source_name else all_features
    val_probabilities = fitted_model.predict_proba(X_val[features])[:, 1]

    for threshold in threshold_values:
        val_predictions = (val_probabilities >= threshold).astype(int)
        threshold_rows.append(
            {
                "Model": model_name,
                "Threshold": threshold,
                "Churn precision": precision_score(y_val, val_predictions, zero_division=0),
                "Churn recall": recall_score(y_val, val_predictions, zero_division=0),
                "Churn F1": f1_score(y_val, val_predictions, zero_division=0),
            }
        )

    precision, recall, thresholds = precision_recall_curve(y_val, val_probabilities)
    eligible = np.flatnonzero(precision[:-1] >= MIN_CHURN_PRECISION)
    if eligible.size == 0:
        raise ValueError(
            f"{model_name} did not reach minimum precision {MIN_CHURN_PRECISION:.2f}."
        )
    best_idx = eligible[np.argmax(recall[:-1][eligible])]
    selected_thresholds[model_name] = thresholds[best_idx]
    print(
        f"{model_name}: threshold={thresholds[best_idx]:.3f}, "
        f"validation precision={precision[best_idx]:.3f}, "
        f"recall={recall[best_idx]:.3f}"
    )

display(pd.DataFrame(threshold_rows).round(3))


## Final test evaluation and sample predictions

The test set is used only after model and threshold selection. The last table compares the same ten held-out customers across all six model variants.


In [ ]:
test_probabilities = {}
test_predictions = {}

for model_name, fitted_model in models_to_compare.items():
    source_name = feature_key_for_reduced.get(model_name)
    features = selected_features_by_model[source_name] if source_name else all_features
    probabilities = fitted_model.predict_proba(X_test[features])[:, 1]
    predictions = (probabilities >= selected_thresholds[model_name]).astype(int)
    test_probabilities[model_name] = probabilities
    test_predictions[model_name] = predictions

    print(f"
--- Held-out test results: {model_name} ---")
    print(f"Threshold: {selected_thresholds[model_name]:.3f}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"ROC AUC: {roc_auc_score(y_test, probabilities):.3f}")
    print(
        classification_report(
            y_test, predictions, labels=[0, 1],
            target_names=["No churn", "Churn"], zero_division=0,
        )
    )
    print(
        "Confusion matrix (actual rows; predicted columns):
",
        confusion_matrix(y_test, predictions, labels=[0, 1]),
    )

sample_index = X_test.head(10).index
sample_results = pd.DataFrame(index=sample_index)
sample_results["ActualChurn"] = y_test.loc[sample_index]
for model_name in models_to_compare:
    sample_results[f"{model_name} predicted"] = test_predictions[model_name][:10]
    sample_results[f"{model_name} probability"] = test_probabilities[model_name][:10]

display(sample_results.round(3))
